In [13]:
import pandas as pd
import re

def process_age_sex(file_path, year):
    df = pd.read_csv(file_path, skiprows=1)

    df = df.rename(columns={
        "Geography": "GEO_ID",
        "Geographic Area Name": "NAME"
    })

    estimate_cols = [c for c in df.columns if str(c).startswith("Estimate")]

    def norm(s):
        return re.sub(r"[^a-z0-9]+", "", str(s).lower())

    def find_cols(sex_text, age_parts):
        matches = []
        for c in estimate_cols:
            c_norm = norm(c)
            if norm(sex_text) in c_norm and any(norm(p) in c_norm for p in age_parts):
                matches.append(c)
        if not matches:
            raise KeyError(f"No match for sex={sex_text}, ages={age_parts}")
        return matches

    age_bins = {
        "18-24": ["18 and 19 years", "20 years", "21 years", "22 to 24 years"],
        "25-29": ["25 to 29 years"],
        "30-34": ["30 to 34 years"],
        "35-39": ["35 to 39 years"],
        "40-44": ["40 to 44 years"],
        "45-49": ["45 to 49 years"],
        "50-54": ["50 to 54 years"],
        "55-59": ["55 to 59 years"],
        "60-64": ["60 and 61 years", "62 to 64 years"],
        "65-69": ["65 and 66 years", "67 to 69 years"],
        "70-74": ["70 to 74 years"],
        "75-79": ["75 to 79 years"],
        "80+": ["80 to 84 years", "85 years and over"]
    }

    results = []

    for sex_label in ["Male", "Female"]:
        for age_label, age_parts in age_bins.items():
            cols = find_cols(sex_label, age_parts)

            vals = pd.concat(
                [pd.to_numeric(df[c], errors="coerce") for c in cols],
                axis=1
            ).sum(axis=1)

            results.append(pd.DataFrame({
                "GEO_ID": df["GEO_ID"],
                "NAME": df["NAME"],
                "sex": sex_label,
                "age": age_label,
                "population": vals
            }))

    df_long = pd.concat(results, ignore_index=True)
    df_long["year"] = year
    df_long["prop"] = df_long["population"] / df_long.groupby("GEO_ID")["population"].transform("sum")

    return df_long

In [15]:
age_2020 = process_age_sex(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2020.B01001-Data.csv", 2020)
age_2021 = process_age_sex(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2021.B01001-Data.csv", 2021)
age_2022 = process_age_sex(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2022.B01001-Data.csv", 2022)
age_2023 = process_age_sex(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2023.B01001-Data.csv", 2023)
age_2024 = process_age_sex(r"C:\Users\kband\Documents\ACS2020-2021\ACSDT5Y2024.B01001-Data.csv", 2024)

In [17]:
age_all = pd.concat([age_2020, age_2021, age_2022, age_2023, age_2024], ignore_index=True)
age_all.to_csv("age_sex_all_2020_2024.csv", index=False)

In [19]:
age_all.groupby(["GEO_ID", "year"])["prop"].sum().unique()

array([1.])

In [21]:
age_all

,GEO_ID,NAME,sex,age,population,year,prop
0,0100000US,United States,Male,18-24,30435736,2020,0.079429
1,0100000US,United States,Male,25-29,23262155,2020,0.060708
2,0100000US,United States,Male,30-34,22223010,2020,0.057996
3,0100000US,United States,Male,35-39,21346055,2020,0.055707
4,0100000US,United States,Male,40-44,20000622,2020,0.052196
...,...,...,...,...,...,...,...
125,0100000US,United States,Female,60-64,11013823,2024,0.027909
126,0100000US,United States,Female,65-69,9825853,2024,0.024899
127,0100000US,United States,Female,70-74,8208903,2024,0.020801
128,0100000US,United States,Female,75-79,5750350,2024,0.014571
